# 01 — EDA Exprés + Feature Store

**Objetivo:** Construir el dataset agregado a nivel `article × week` que alimentará al modelo de demanda. Sin EDA decorativo — solo lo necesario para validar decisiones de modelado.

**Tiempo estimado:** 1.5 horas (incluyendo la carga inicial de ~31M filas).

**Outputs:**
- `data/processed/weekly_sales.parquet` — series temporales por producto
- `data/processed/article_features.parquet` — metadata limpia de productos
- 3 insights de negocio para el README

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loaders import load_articles, load_transactions
from src.config import PROCESSED_DIR, ensure_dirs

ensure_dirs()
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

## 1. Cargar transacciones (los 31M completos)

Esto tarda ~1-2 min. Si el ordenador va justo de RAM, podemos limitar luego.

In [ ]:
%%time
transactions = load_transactions()  # carga completa
print(f'Shape: {transactions.shape}')
print(f'Memoria: {transactions.memory_usage(deep=True).sum() / 1e9:.2f} GB')

In [ ]:
articles = load_articles()
print(f'Articles: {articles.shape}')

## 2. Insight #1 — Estacionalidad mensual

**Pregunta:** ¿Hay patrones estacionales claros que el modelo deba capturar?

In [ ]:
# Volumen mensual de transacciones
monthly = transactions.groupby(transactions['t_dat'].dt.to_period('M')).size()

fig, ax = plt.subplots(figsize=(14, 4))
monthly.plot(ax=ax, marker='o')
ax.set_title('Volumen mensual de transacciones (sept 2018 – sept 2020)')
ax.set_ylabel('# transacciones')
ax.set_xlabel('Mes')
plt.tight_layout()
plt.savefig('../reports/figures/01_monthly_seasonality.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nMáximos:')
print(monthly.nlargest(5))
print('\nMínimos:')
print(monthly.nsmallest(3))

**👉 Lo que esperamos ver:**
- Picos en junio/julio (rebajas verano) y noviembre/diciembre (Black Friday + Navidad)
- Posible bajón en marzo-abril 2020 por COVID — esto será **una historia muy potente para el README**

## 3. Insight #2 — Concentración de ventas (regla 80/20)

**Pregunta:** ¿Pocos productos generan la mayoría de las ventas? Esto valida el enfoque jerárquico del modelo.

In [ ]:
sales_per_article = transactions.groupby('article_id').size().sort_values(ascending=False)

cumulative_pct = sales_per_article.cumsum() / sales_per_article.sum() * 100
n_articles_pct = np.arange(1, len(sales_per_article) + 1) / len(sales_per_article) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_articles_pct, cumulative_pct, linewidth=2)
ax.axhline(80, color='red', linestyle='--', alpha=0.5)
ax.axvline(20, color='red', linestyle='--', alpha=0.5)
ax.set_title('Curva de Pareto: ¿qué % de productos genera qué % de ventas?')
ax.set_xlabel('% de productos (ordenados de más a menos vendidos)')
ax.set_ylabel('% acumulado de ventas')
plt.tight_layout()
plt.savefig('../reports/figures/02_pareto.png', dpi=120, bbox_inches='tight')
plt.show()

# Cifras exactas
for pct in [10, 20, 50]:
    n = int(len(sales_per_article) * pct / 100)
    pct_sales = cumulative_pct.iloc[n - 1]
    print(f'El top {pct}% de productos ({n:,} SKUs) genera el {pct_sales:.1f}% de las ventas')

**👉 Insight de negocio:** Si el top 20% de productos genera el ~70-80% de las ventas, **el resto es "long tail"** — productos con muy pocas ventas y alta variabilidad. Esto justifica usar features jerárquicas (medias por categoría) en el modelo.

## 4. Insight #3 — Distribución de precios

**Pregunta:** ¿Hay outliers extremos que distorsionen al modelo?

In [ ]:
# Nota: los precios en H&M están normalizados (no son EUR reales)
# Los multiplicaremos por un factor para convertirlos a una escala razonable luego
price_stats = transactions['price'].describe(percentiles=[0.01, 0.5, 0.95, 0.99])
print(price_stats)

fig, ax = plt.subplots(figsize=(10, 4))
transactions['price'].plot(kind='hist', bins=100, ax=ax, alpha=0.7)
ax.set_xlim(0, transactions['price'].quantile(0.99))
ax.set_title('Distribución de precios (normalizados, hasta percentil 99)')
ax.set_xlabel('precio normalizado')
plt.tight_layout()
plt.savefig('../reports/figures/03_price_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 🔨 Construir el feature store: `article × week`

Aquí está el corazón del bloque. Vamos a:
1. Crear columna `week` (semana del año)
2. Agregar transacciones por (article_id, week) → ventas semanales
3. Hacer join con metadata de productos
4. Guardar como parquet

In [ ]:
%%time
# Truncar la fecha al lunes de cada semana
transactions['week'] = transactions['t_dat'].dt.to_period('W').dt.start_time

# Agregar a nivel article × week
weekly = (
    transactions
    .groupby(['article_id', 'week'], observed=True)
    .agg(
        units_sold=('article_id', 'size'),
        avg_price=('price', 'mean'),
    )
    .reset_index()
)

print(f'Filas en weekly: {len(weekly):,}')
weekly.head()

### Solo conservamos productos con suficiente histórico

Para entrenar un modelo de demanda con sentido, necesitamos al menos **8 semanas** de histórico por producto. Productos efímeros (1-2 semanas en catálogo) son ruido para el modelo.

In [ ]:
weeks_per_article = weekly.groupby('article_id', observed=True).size()
valid_articles = weeks_per_article[weeks_per_article >= 8].index

print(f'Productos totales: {len(weeks_per_article):,}')
print(f'Productos con >=8 semanas: {len(valid_articles):,} ({len(valid_articles)/len(weeks_per_article)*100:.1f}%)')

weekly = weekly[weekly['article_id'].isin(valid_articles)].copy()
print(f'\nFilas finales: {len(weekly):,}')

### Añadir metadata de productos

Solo nos quedamos con las columnas que de verdad usaremos en los modelos.

In [ ]:
feature_cols = [
    'article_id',
    'product_type_name',
    'product_group_name',
    'graphical_appearance_name',
    'colour_group_name',
    'perceived_colour_master_name',
    'department_name',
    'index_group_name',
    'section_name',
    'garment_group_name',
]

article_features = articles[feature_cols].copy()
article_features = article_features[article_features['article_id'].isin(valid_articles)]
print(f'Article features shape: {article_features.shape}')
article_features.head()

## 6. Guardar todo en `data/processed/`

In [ ]:
weekly_path = PROCESSED_DIR / 'weekly_sales.parquet'
features_path = PROCESSED_DIR / 'article_features.parquet'

weekly.to_parquet(weekly_path, index=False)
article_features.to_parquet(features_path, index=False)

print(f'✅ Guardado: {weekly_path}')
print(f'   {len(weekly):,} filas, {weekly_path.stat().st_size / 1e6:.1f} MB')
print(f'\n✅ Guardado: {features_path}')
print(f'   {len(article_features):,} filas, {features_path.stat().st_size / 1e6:.1f} MB')

## 7. Resumen ejecutivo

Apunta los números clave para el README final:

| Métrica | Valor |
|---|---|
| Total transacciones | _completa_ |
| Periodo cubierto | sept 2018 – sept 2020 |
| Productos con histórico válido (>=8 sem) | _completa_ |
| Filas en feature store semanal | _completa_ |
| Concentración: top 20% productos generan | _X_% de ventas |

✅ Bloque 1 completado. Siguiente: `02_demand_model.ipynb`